# Fine-tune a market-research agent brain with Unsloth

Fine-tunes **Qwen2.5-3B-Instruct** with LoRA on:

- **NousResearch/hermes-function-calling-v1**: teaches reliable `<tool_call>{json}</tool_call>` tool use. That is the same format Ollama's Qwen2.5 template parses, so the result plugs straight into `market_research_agent/`.
- **gbharti/finance-alpaca**: finance and markets vocabulary and reasoning.

Then it exports a **Q4_K_M GGUF** (~2 GB), which runs in Ollama on a 4 GB GPU.

**Runtime:** Colab → Runtime → Change runtime type → **T4 GPU**. About 45-60 min end to end.

In [ ]:
%%capture
!pip install unsloth

## 1. Load the base model (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,  # auto: float16 on T4, bfloat16 on Ampere+
)

## 2. Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",  # big VRAM saving for long tool-call transcripts
    random_state=3407,
)

## 3. Build the training mix

Both datasets are converted to `[{"role", "content"}]` conversations:

- Hermes `tool` turns become `user` turns wrapped in `<tool_response>`, which is exactly how Qwen2.5 renders tool results.
- Finance answers shorter than 200 characters are dropped, so the model learns to answer in depth.

In [ ]:
from datasets import load_dataset, concatenate_datasets

N_TOOL, N_FINANCE = 3000, 2000
ROLE_MAP = {"system": "system", "human": "user", "gpt": "assistant", "tool": "user"}


def hermes_to_messages(example):
    messages = []
    for turn in example["conversations"]:
        role, content = ROLE_MAP[turn["from"]], turn["value"]
        if turn["from"] == "tool" and "<tool_response>" not in content:
            content = f"<tool_response>\n{content}\n</tool_response>"
        messages.append({"role": role, "content": content})
    return {"messages": messages}


def finance_to_messages(example):
    prompt = example["instruction"] + (f"\n\n{example['input']}" if example["input"] else "")
    return {"messages": [
        {"role": "system", "content": "You are a meticulous financial and market research analyst."},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": example["output"]},
    ]}


tool_ds = load_dataset("NousResearch/hermes-function-calling-v1", "func_calling_singleturn", split="train")
tool_ds = tool_ds.shuffle(seed=3407).select(range(min(N_TOOL, len(tool_ds))))
tool_ds = tool_ds.map(hermes_to_messages, remove_columns=tool_ds.column_names)

fin_ds = load_dataset("gbharti/finance-alpaca", split="train")
fin_ds = fin_ds.filter(lambda x: len(x["output"]) > 200).shuffle(seed=3407)
fin_ds = fin_ds.select(range(min(N_FINANCE, len(fin_ds))))
fin_ds = fin_ds.map(finance_to_messages, remove_columns=fin_ds.column_names)

dataset = concatenate_datasets([tool_ds, fin_ds]).shuffle(seed=3407)
print(dataset)
print(dataset[0]["messages"][:3])

## 4. Apply the Qwen2.5 chat template

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")


def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
                     for m in batch["messages"]]}


dataset = dataset.map(to_text, batched=True)
# Drop examples that would be truncated mid tool-call
dataset = dataset.filter(lambda x: len(tokenizer(x["text"]).input_ids) <= MAX_SEQ_LEN)
print(len(dataset), "examples")
print(dataset[0]["text"][:1500])

## 5. Train (loss on assistant turns only)

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=250,          # ~45 min on a T4; set num_train_epochs=1 instead for a full pass
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
# Only learn from what the assistant says (tool calls + answers), not from prompts/tool outputs.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

stats = trainer.train()
print(f"{stats.metrics['train_runtime'] / 60:.1f} min, final loss {stats.metrics['train_loss']:.3f}")

## 6. Sanity check: does it call tools?

In [ ]:
FastLanguageModel.for_inference(model)

tools = [{
    "type": "function",
    "function": {
        "name": "stock_quote",
        "description": "Company profile, price, valuation multiples, margins and analyst targets.",
        "parameters": {"type": "object", "properties": {"ticker": {"type": "string"}}, "required": ["ticker"]},
    },
}]
messages = [{"role": "user", "content": "What's NVIDIA's current forward P/E and analyst target?"}]

inputs = tokenizer.apply_chat_template(messages, tools=tools, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.3, do_sample=True)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=False))
# Expect something like: <tool_call>\n{"name": "stock_quote", "arguments": {"ticker": "NVDA"}}\n</tool_call>

## 7. Save the LoRA adapter and export GGUF for Ollama

In [ ]:
model.save_pretrained("market-researcher-lora")
tokenizer.save_pretrained("market-researcher-lora")

model.save_pretrained_gguf("market-researcher", tokenizer, quantization_method="q4_k_m")

import glob
gguf_files = glob.glob("**/*.gguf", recursive=True)
print(gguf_files)

In [ ]:
# Download the GGUF to your machine (or push to the Hugging Face Hub, below).
from google.colab import files
files.download(max(gguf_files, key=lambda p: "Q4_K_M" in p.upper()))

In [ ]:
# Optional: push to the Hugging Face Hub instead of downloading
# model.push_to_hub_gguf("your-hf-username/market-researcher", tokenizer, quantization_method="q4_k_m", token="hf_...")

## 8. Run it locally with Ollama

On your machine, from the repo:

```bash
cd unsloth_finetuning
./make_ollama_model.sh ~/Downloads/<downloaded-file>.gguf market-researcher
ollama run market-researcher "Explain free cash flow yield in two sentences."
```

The script reuses the official `qwen2.5:3b` Ollama template, so tool calling works with `market_research_agent/research.py`.